# SeineCrops — Sprint S3 : Classification des cultures

Classification supervisée des parcelles RPG à partir des séries
temporelles Sentinel-2 agrégées en S2. Le modèle reçoit en entrée
les **704 features** de `derived.s2_parcelles_monthly` (11 variables
× 4 statistiques × 16 mois) pivotées en format wide (une ligne par
parcelle). Les codes cultures RPG sont regroupés en **7–8 classes**
(blé tendre, orge, colza, maïs, betterave, lin, prairies, autres).

---

## Approche

**Baseline** : Random Forest (scikit-learn), choisi pour sa robustesse
aux features non normalisées et sa lisibilité (feature importances).
Le split est **spatial par blocs** — pas de split aléatoire, qui
créerait une fuite spatiale entre parcelles voisines partageant le
même contexte pédo-climatique et des profils temporels corrélés.

**Évaluation** : matrice de confusion, F1 macro et F1 par classe,
avec attention portée aux classes minoritaires (lin, betterave).
Cible indicative : **F1 macro ≥ 0,85** sur les grandes cultures.

**Option DL** : 1D-CNN ou LSTM sur la dimension temporelle (16 pas
de temps), à envisager si la baseline Random Forest ne satisfait pas
les critères de validation.

---

## Structure du notebook

| Section | Contenu |
|---------|---------|
| 4.1 | Préparation du feature set — pivot wide, regroupement des cultures, diagnostic des valeurs manquantes |
| 4.1bis | Règle de décision sur la complétude temporelle — exclure / imputer / conserver à partir de `s2_parcelles_completude`, correction des ancrages d'interpolation à distance > 1 mois |
| 4.2 | Split spatial par blocs — découpage géographique train / test sans fuite spatiale |
| 4.3 | Entraînement Random Forest — baseline, tuning hyperparamètres, évaluation (matrice de confusion, F1 macro / par classe, feature importances) |
| 4.4 | Sauvegarde des prédictions — persistance dans `derived.parcelles_classification` (préparation S5) |

---

## Références

- [scikit-learn — RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [Spatial cross-validation — blocage géographique (Roberts et al., 2017)](https://doi.org/10.1111/ecog.02881)
- `cadrage/methode.md` — §S3 Classification
- `03_series_s2.ipynb` — sprint S2, `derived.s2_parcelles_monthly` (feature set source)
- `01_ingestion_rpg.ipynb` — sprint S1, `derived.rpg_parcelles_aoi` (cultures déclarées)

### 4.1 — Préparation du feature set

Construction de la matrice d'entrée du modèle à partir de
`derived.s2_parcelles_monthly` (format long : une ligne par
parcelle × mois × variable) et de `derived.rpg_parcelles_aoi`
(culture déclarée).

**Pivot wide** : la table longue est pivotée en une matrice
(parcelles × features) de **704 colonnes** — une par combinaison
`{variable}_{stat}_{mois}` (ex. `ndvi_mean_2024-06`). Chaque ligne
correspond à une parcelle identifiée par `id_parcel`.

**Regroupement des cultures** : les codes `code_group` du RPG sont
regroupés en 7–8 classes cibles adaptées à l'openfield normand :
blé tendre, orge, colza, maïs, betterave, lin, prairies, autres.
Le regroupement est défini dans un dictionnaire explicite pour
assurer la traçabilité et permettre un ajustement ultérieur des
seuils de classe.

**Diagnostic NaN** : les 2 751 parcelles sans pixel capturé à 20 m
(identifiées en S2.4) sont absentes de la table longue et donc du
pivot. Pour les parcelles présentes, les valeurs manquantes
(mois × variable sans composite valide) sont diagnostiquées :
taux de NaN global et par variable/mois, distribution par classe.
La stratégie d'imputation (ou d'exclusion) est décidée après
diagnostic.

In [ ]:
# ── Imports (communs à toutes les sections) ───────────────────────────
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# ── Racine du projet ──────────────────────────────────────────────────
def find_project_root(marker: str = ".projectroot") -> Path:
    here = Path().resolve()
    for parent in [here, *here.parents]:
        if (parent / marker).exists() or (parent / ".git").exists():
            return parent
    raise FileNotFoundError("Racine du projet introuvable")

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

# ── Connexion PostGIS ─────────────────────────────────────────────────
PG_PARAMS = {
    "host": os.getenv("PG_HOST", "localhost"),
    "port": int(os.getenv("PG_PORT", 5432)),
    "dbname": os.getenv("PG_DBNAME", "seinecrops"),
    "user": os.getenv("PG_USER", "postgres"),
    "password": os.getenv("PG_PASSWORD"),
}

In [ ]:
# --- Chargement depuis PostGIS ---------------------------------------

conn = psycopg2.connect(**PG_PARAMS)

# Feature set long : ~14 M lignes
sql_features = """
    SELECT id_parcel, mois, variable, mean, std, p10, p90
    FROM derived.s2_parcelles_monthly
    ORDER BY id_parcel, mois, variable
"""
df_long = pd.read_sql(sql_features, conn)

# Labels RPG : code_group par parcelle
sql_labels = """
    SELECT id_parcel, code_group
    FROM derived.rpg_parcelles_aoi
"""
df_labels = pd.read_sql(sql_labels, conn)

conn.close()

print(f"Features long : {len(df_long):>12,} lignes")
print(f"Parcelles dans features : {df_long['id_parcel'].nunique():>7,}")
print(f"Parcelles RPG : {len(df_labels):>12,} parcelles")

In [ ]:
# --- Pivot long → wide -----------------------------------------------

STATS = ["mean", "std", "p10", "p90"]

# Melt stats en une seule colonne (stat, value) pour pivot propre
df_melt = df_long.melt(
    id_vars=["id_parcel", "mois", "variable"],
    value_vars=STATS,
    var_name="stat",
    value_name="value",
)

# Clé de colonne : variable_stat_mois (ex. ndvi_mean_2024-06)
df_melt["feature"] = (
    df_melt["variable"] + "_" + df_melt["stat"] + "_" + df_melt["mois"]
)

# Pivot : une ligne par parcelle, une colonne par feature
df_wide = df_melt.pivot(
    index="id_parcel", columns="feature", values="value"
)
df_wide.columns.name = None

print(f"Matrice wide : {df_wide.shape[0]:,} parcelles × {df_wide.shape[1]} features")
assert df_wide.shape[1] == 704, f"Attendu 704 features, obtenu {df_wide.shape[1]}"

In [ ]:
# --- Regroupement des cultures (v3 : 8 classes) ----------------------

conn = psycopg2.connect(**PG_PARAMS)

sql_labels = """
    SELECT id_parcel, code_group::int AS code_group, code_cultu
    FROM derived.rpg_parcelles_aoi
"""
df_labels = pd.read_sql(sql_labels, conn)
conn.close()

# Dédupliquer les 6 id_parcel en doublon dans le RPG
df_labels = df_labels.drop_duplicates(subset="id_parcel", keep="first")

# Mapping code_group → classe cible (v3 : 8 classes)
GROUP_MAP = {
    1:  "cereales_hiver",      # Blé tendre
    3:  "cereales_hiver",      # Orge
    2:  "mais",                # Maïs grain et ensilage
    5:  "colza",               # Colza
    9:  "lin",                 # Plantes à fibres (≈ lin fibre en Normandie)
    18: "prairie",             # Prairies permanentes
    19: "prairie",             # Prairies temporaires
    25: "legumes_fleurs",      # Légumes ou fleurs
}

# Mapping en deux temps : d'abord code_group, puis exception betterave
df_labels["classe"] = df_labels["code_group"].map(GROUP_MAP).fillna("autres")
df_labels.loc[df_labels["code_cultu"] == "BTN", "classe"] = "betterave"

# Supprimer la colonne classe si déjà présente (ré-exécution)
df_wide = df_wide.drop(columns=["classe"], errors="ignore")

# Jointure avec le feature set
df_wide = df_wide.join(
    df_labels.set_index("id_parcel")[["classe"]]
)

print(f"Parcelles avec classe assignée : {df_wide['classe'].notna().sum():,} / {len(df_wide):,}")
print(df_wide["classe"].value_counts().to_string())

In [ ]:
# --- Diagnostic valeurs manquantes -----------------------------------

n_total = df_wide.shape[0] * 704
n_nan = df_wide.drop(columns=["classe"]).isna().sum().sum()

print(f"NaN : {n_nan:,} / {n_total:,} ({100 * n_nan / n_total:.2f} %)\n")

# Taux de NaN par mois
feature_cols = [c for c in df_wide.columns if c != "classe"]
nan_by_month = {}
for col in feature_cols:
    mois = col.rsplit("_", 1)[-1]       # ndvi_mean_2024-06 → 2024-06
    nan_by_month.setdefault(mois, []).append(df_wide[col].isna().sum())

print("NaN par mois (total sur toutes features) :")
for mois in sorted(nan_by_month):
    total = sum(nan_by_month[mois])
    pct = 100 * total / (len(nan_by_month[mois]) * len(df_wide))
    print(f"  {mois} : {total:>8,}  ({pct:.2f} %)")

### 4.1bis — Règle de décision sur la complétude temporelle

La table `derived.s2_parcelles_completude` (3.5) porte, par
`id_parcel × mois`, le taux de couverture spatiale (`pct_pixels_couverts`)
ayant contribué au composite du mois. Elle permet de trancher, pour
chaque cellule de la matrice de features, entre trois actions :
**exclure** (0 % de couverture — NaN structurel, laissé tel quel),
**imputer** (couverture < 50 % — valeur peu fiable, remplacée par
interpolation temporelle linéaire sur les mois voisins de la même
variable), **conserver** (couverture ≥ 50 % — valeur inchangée).

In [ ]:
# --- Chargement de la table de complétude (3.5) ------------------------

conn = psycopg2.connect(**PG_PARAMS)
sql_completude = """
    SELECT id_parcel, mois, n_dates_valides_moy, pct_pixels_couverts
    FROM derived.s2_parcelles_completude
"""
df_completude = pd.read_sql(sql_completude, conn)
conn.close()

print(f"Complétude : {len(df_completude):,} lignes (id_parcel × mois)")

# --- Règle de décision ---------------------------------------------------
SEUIL_COUVERTURE = 0.50

def qc_action(pct):
    if pct == 0:
        return "exclure"
    elif pct < SEUIL_COUVERTURE:
        return "imputer"
    return "conserver"

df_completude["qc_action"] = df_completude["pct_pixels_couverts"].apply(qc_action)

print("\nRépartition globale (parcelle × mois) :")
print(df_completude["qc_action"].value_counts().to_string())

print("\nRépartition par mois :")
print(pd.crosstab(df_completude["mois"], df_completude["qc_action"]).to_string())

In [ ]:
# --- Diagnostic : distance d'ancrage de l'interpolation ---------------

MOIS_IDX = {m: i for i, m in enumerate(MOIS_ORDER)}

# Ne traiter que les parcelles ayant au moins une cellule "imputer"
has_impute = (tier_wide == "imputer").any(axis=1)
sous_tier = tier_wide[has_impute]

rows_diag = []
for id_parcel, row in sous_tier.iterrows():
    for mois in MOIS_ORDER:
        if row[mois] != "imputer":
            continue
        i = MOIS_IDX[mois]

        # Voisin valide (tier != "exclure") le plus proche à gauche
        dist_gauche = next(
            (i - j for j in range(i - 1, -1, -1) if row[MOIS_ORDER[j]] != "exclure"),
            None,
        )
        # Voisin valide le plus proche à droite
        dist_droite = next(
            (j - i for j in range(i + 1, len(MOIS_ORDER)) if row[MOIS_ORDER[j]] != "exclure"),
            None,
        )

        candidats = [d for d in (dist_gauche, dist_droite) if d is not None]
        dist_min = min(candidats) if candidats else None

        rows_diag.append({
            "id_parcel": id_parcel, "mois": mois,
            "dist_gauche": dist_gauche, "dist_droite": dist_droite,
            "dist_min": dist_min,
        })

df_diag = pd.DataFrame(rows_diag)

print(f"Cellules 'imputer' analysées : {len(df_diag):,}")
print(f"Sans aucun ancrage valide (des deux côtés) : {df_diag['dist_min'].isna().sum():,}")

print("\nDistribution de la distance minimale d'ancrage (en mois) :")
print(df_diag["dist_min"].value_counts().sort_index().to_string())

print(f"\nAncrées à > 1 mois : {(df_diag['dist_min'] > 1).sum():,} / {len(df_diag):,}")

In [ ]:
# --- Correction du tier : repasser en "exclure" les ancrages > 1 mois --

cellules_a_corriger = df_diag.loc[df_diag["dist_min"] > 1, ["id_parcel", "mois"]]

print(f"Cellules repassées en 'exclure' (ancrage > 1 mois) : {len(cellules_a_corriger):,}")

for _, r in cellules_a_corriger.iterrows():
    tier_wide.loc[r["id_parcel"], r["mois"]] = "exclure"

# Mise à jour de df_completude en cohérence (pour la trace / methode.md)
idx_correction = df_completude.set_index(["id_parcel", "mois"]).index.isin(
    cellules_a_corriger.set_index(["id_parcel", "mois"]).index
)
df_completude.loc[idx_correction, "qc_action"] = "exclure"

print("\nRépartition finale :")
print(df_completude["qc_action"].value_counts().to_string())

In [ ]:
# --- Application de la règle : interpolation temporelle ciblée ---------

tier_wide = df_completude.pivot(index="id_parcel", columns="mois", values="qc_action")
tier_wide = tier_wide.reindex(df_wide.index)

MOIS_ORDER = sorted(tier_wide.columns)
variables = sorted({c.split("_")[0] for c in df_wide.columns if c != "classe"})

n_impute_total = 0

for var in variables:
    for stat in STATS:
        cols_vs = [f"{var}_{stat}_{m}" for m in MOIS_ORDER if f"{var}_{stat}_{m}" in df_wide.columns]
        if not cols_vs:
            continue

        sub = df_wide[cols_vs]
        interpolated = sub.interpolate(axis=1, limit_direction="both")

        mois_present = [m for m in MOIS_ORDER if f"{var}_{stat}_{m}" in df_wide.columns]
        mask_impute = (tier_wide[mois_present] == "imputer")
        mask_impute.columns = cols_vs

        n_impute_total += int(mask_impute.sum().sum())
        df_wide.loc[:, cols_vs] = sub.where(~mask_impute, interpolated)

n_nan_after = df_wide.drop(columns=["classe"], errors="ignore").isna().sum().sum()

print(f"Valeurs imputées (interpolation temporelle) : {n_impute_total:,}")
print(f"NaN résiduels (tier 'exclure') : {n_nan_after:,}")

### 4.2 — Split spatial par blocs

Découpage train / test réduisant la fuite spatiale. Un split
aléatoire placerait des parcelles voisines — partageant le même
contexte pédo-climatique et des profils temporels corrélés — de
part et d'autre de la frontière train/test, gonflant
artificiellement les métriques d'évaluation.

**Méthode** : l'emprise de l'AOI est découpée en blocs
géographiques réguliers (grille carrée). Chaque bloc est affecté
intégralement au train ou au test, de sorte que les parcelles
d'un même voisinage local restent ensemble. Aux frontières entre
blocs, des parcelles train et test se jouxtent — l'autocorrélation
est réduite, pas éliminée. La taille de bloc est un compromis :
trop petite, l'effet de bloc s'efface ; trop grande, le nombre
de blocs diminue et la variance du split augmente.

**Ratio cible** : ~80 % train / 20 % test en nombre de parcelles,
avec vérification que chaque classe conserve une représentation
suffisante dans les deux ensembles.

**Géométrie** : les centroïdes des parcelles (issus de
`derived.rpg_parcelles_aoi`) servent à affecter chaque parcelle
à un bloc. La grille est alignée sur l'emprise de l'AOI en
EPSG:2154.

In [ ]:
# --- Centroïdes des parcelles ----------------------------------------

conn = psycopg2.connect(**PG_PARAMS)

sql_centroids = """
    SELECT id_parcel,
           ST_X(ST_Centroid(geom)) AS cx,
           ST_Y(ST_Centroid(geom)) AS cy
    FROM derived.rpg_parcelles_aoi
"""
df_centr = pd.read_sql(sql_centroids, conn)
conn.close()

df_centr = df_centr.drop_duplicates(subset="id_parcel", keep="first")

# Ne garder que les parcelles présentes dans le feature set
df_centr = df_centr[df_centr["id_parcel"].isin(df_wide.index)]

print(f"Centroïdes : {len(df_centr):,} parcelles")
print(f"Emprise X : {df_centr['cx'].min():.0f} → {df_centr['cx'].max():.0f} m")
print(f"Emprise Y : {df_centr['cy'].min():.0f} → {df_centr['cy'].max():.0f} m")

In [ ]:
# --- Split spatial par blocs -----------------------------------------

BLOCK_SIZE = 10_000  # 10 km
TEST_RATIO = 0.20
SEED = 42

# Affecter chaque parcelle à un bloc (identifié par indices grille)
x_min, y_min = df_centr["cx"].min(), df_centr["cy"].min()
df_centr["block_x"] = ((df_centr["cx"] - x_min) // BLOCK_SIZE).astype(int)
df_centr["block_y"] = ((df_centr["cy"] - y_min) // BLOCK_SIZE).astype(int)
df_centr["block_id"] = df_centr["block_x"].astype(str) + "_" + df_centr["block_y"].astype(str)

# Tirage aléatoire des blocs test
blocks = df_centr["block_id"].unique()
rng = np.random.default_rng(SEED)
n_test_blocks = max(1, int(len(blocks) * TEST_RATIO))
test_blocks = set(rng.choice(blocks, size=n_test_blocks, replace=False))

df_centr["split"] = df_centr["block_id"].apply(
    lambda b: "test" if b in test_blocks else "train"
)

# Reporter le split sur df_wide (idempotent)
df_wide = df_wide.drop(columns=["split"], errors="ignore")
df_wide = df_wide.join(
    df_centr.set_index("id_parcel")[["split"]],
    on="id_parcel",
)

n_train = (df_wide["split"] == "train").sum()
n_test = (df_wide["split"] == "test").sum()
print(f"Blocs : {len(blocks)} total, {n_test_blocks} test")
print(f"Train : {n_train:,} ({100*n_train/len(df_wide):.1f} %)")
print(f"Test  : {n_test:,}  ({100*n_test/len(df_wide):.1f} %)")

In [ ]:
# --- Représentation par classe dans train / test ---------------------

ct = pd.crosstab(df_wide["classe"], df_wide["split"])
ct["total"] = ct.sum(axis=1)
ct["pct_test"] = (100 * ct["test"] / ct["total"]).round(1)
ct = ct.sort_values("total", ascending=False)

print(ct.to_string())

### 4.3 — Entraînement Random Forest

Baseline de classification : Random Forest scikit-learn sur le
feature set de 704 colonnes.

**Préparation** : les matrices `X_train`, `X_test` (features) et
`y_train`, `y_test` (classe cible) sont construites à partir du
split spatial défini en 4.2. Aucune normalisation n'est nécessaire
— Random Forest est invariant aux échelles des features.

**Hyperparamètres** : un premier entraînement utilise des
hyperparamètres par défaut, suivi d'une recherche par
`RandomizedSearchCV` (validation croisée sur le train uniquement)
pour ajuster `n_estimators`, `max_depth`, `min_samples_leaf` et
`max_features`. Le jeu test reste intouché jusqu'à l'évaluation
finale.

**Feature importances** : le modèle entraîné fournit un classement
des features les plus discriminantes, permettant d'identifier les
variables spectrales et les mois les plus utiles à la séparation
des classes.

In [ ]:
# --- Matrices X / y -------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

feature_cols = [c for c in df_wide.columns if c not in ("classe", "split")]

train_mask = df_wide["split"] == "train"
test_mask = df_wide["split"] == "test"

X_train = df_wide.loc[train_mask, feature_cols].values
X_test = df_wide.loc[test_mask, feature_cols].values
y_train = df_wide.loc[train_mask, "classe"].to_numpy()
y_test = df_wide.loc[test_mask, "classe"].to_numpy()

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"Classes : {np.unique(y_train).tolist()}")

In [ ]:
# --- Random Forest baseline ------------------------------------------

rf_base = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_leaf=5,
    max_features="sqrt",
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
rf_base.fit(X_train, y_train)

score_train = rf_base.score(X_train, y_train)
score_test = rf_base.score(X_test, y_test)

print(f"Accuracy train : {score_train:.4f}")
print(f"Accuracy test  : {score_test:.4f}")

In [ ]:
# --- Matrice de confusion baseline -----------------------------------

from sklearn.metrics import confusion_matrix, classification_report

y_pred_base = rf_base.predict(X_test)

classes = sorted(np.unique(y_train))
cm = confusion_matrix(y_test, y_pred_base, labels=classes)

# Affichage formaté
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print("Matrice de confusion (lignes = réel, colonnes = prédit) :\n")
print(cm_df.to_string())
print(f"\n{classification_report(y_test, y_pred_base, digits=3)}")

In [ ]:
# --- RandomizedSearchCV ----------------------------------------------

param_dist = {
    "n_estimators": [200, 400, 600],
    "max_depth": [20, 30, 40],
    "min_samples_leaf": [2, 5, 10],
    "max_features": ["sqrt", 0.1, 0.2],
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=20,
    scoring="f1_macro",
    cv=3,
    random_state=SEED,
    n_jobs=1,       # parallélisme au niveau du RF, pas du CV
    verbose=2,      # affiche le timing de chaque fit
)
search.fit(X_train, y_train)

print(f"\nMeilleur F1 macro (CV) : {search.best_score_:.4f}")
print(f"Meilleurs paramètres  : {search.best_params_}")

In [ ]:
# --- Évaluation modèle tuné -----------------------------------------

rf = search.best_estimator_

score_train_tuned = rf.score(X_train, y_train)
score_test_tuned = rf.score(X_test, y_test)

y_pred = rf.predict(X_test)

classes = sorted(np.unique(y_train))
cm = confusion_matrix(y_test, y_pred, labels=classes)
cm_df = pd.DataFrame(cm, index=classes, columns=classes)

print(f"Accuracy train : {score_train_tuned:.4f}")
print(f"Accuracy test  : {score_test_tuned:.4f}")
print("\nMatrice de confusion (lignes = réel, colonnes = prédit) :\n")
print(cm_df.to_string())
print(f"\n{classification_report(y_test, y_pred, digits=3)}")

# Top 20 features
feature_cols = [c for c in df_wide.columns if c not in ("classe", "split")]
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)
print("\nTop 20 features :\n")
for feat, imp in top20.items():
    print(f"  {feat:<30s} {imp:.4f}")

### 4.3bis — Test rapide : features temporelles ciblées

Ajout de 3 features à faible coût, visant spécifiquement la confusion
`autres` / `prairie` : amplitude NDVI sur la saison (`s2_parcelles_ndvi_dates`),
date du maximum NDVI, pente NDVI mai → août (à partir des composites
mensuels déjà disponibles). Réentraînement de `rf_base` (mêmes
hyperparamètres par défaut, même split spatial) sur le set augmenté,
comparaison ciblée du F1 `autres` et de sa confusion avec `prairie`.

In [ ]:
# --- Chargement de s2_parcelles_ndvi_dates (3.6) ------------------------

conn = psycopg2.connect(**PG_PARAMS)
sql_ndvi_dates = """
    SELECT id_parcel, date, mean, std, n_pixels
    FROM derived.s2_parcelles_ndvi_dates
"""
df_ndvi_dates = pd.read_sql(sql_ndvi_dates, conn)
conn.close()

df_ndvi_dates["date"] = pd.to_datetime(df_ndvi_dates["date"])

print(f"NDVI par date : {len(df_ndvi_dates):,} lignes, "
      f"{df_ndvi_dates['id_parcel'].nunique():,} parcelles")
print(f"n_pixels : min={df_ndvi_dates['n_pixels'].min()}, "
      f"médiane={df_ndvi_dates['n_pixels'].median():.0f}")

In [ ]:
# --- Filtre de fiabilité (cohérent avec la règle 4.1bis) ---------------

N_PIXELS_MIN = 5  # seuil arbitraire, à ajuster selon la distribution ci-dessus

n_avant = len(df_ndvi_dates)
df_ndvi_dates = df_ndvi_dates[df_ndvi_dates["n_pixels"] >= N_PIXELS_MIN]
print(f"Lignes retenues (n_pixels >= {N_PIXELS_MIN}) : "
      f"{len(df_ndvi_dates):,} / {n_avant:,} "
      f"({100*len(df_ndvi_dates)/n_avant:.1f} %)")

In [ ]:
# --- Amplitude NDVI et date du maximum -----------------------------------

WINDOW_START = pd.Timestamp("2023-09-01")  # début de la fenêtre d'observation

amplitude = df_ndvi_dates.groupby("id_parcel")["mean"].agg(
    ndvi_max_saison="max", ndvi_min_saison="min"
)
amplitude["amplitude_ndvi"] = amplitude["ndvi_max_saison"] - amplitude["ndvi_min_saison"]

idx_max = df_ndvi_dates.groupby("id_parcel")["mean"].idxmax()
df_date_max = df_ndvi_dates.loc[idx_max, ["id_parcel", "date"]].set_index("id_parcel")
df_date_max["jour_max_ndvi"] = (df_date_max["date"] - WINDOW_START).dt.days

features_temporelles = amplitude[["amplitude_ndvi"]].join(df_date_max[["jour_max_ndvi"]])

print(f"Features calculées pour {len(features_temporelles):,} parcelles")
print(features_temporelles.describe().to_string())

In [ ]:
# --- Diagnostic : parcelles manquantes ------------------------------------

parcelles_completes = set(df_wide.index) & set(df_ndvi_dates["id_parcel"].unique())
parcelles_sans_date_valide = set(df_wide.index) - set(df_ndvi_dates["id_parcel"].unique())
# (avant filtre n_pixels : recharger df_ndvi_dates non filtré serait nécessaire pour un diagnostic complet,
#  ici on ne peut distinguer qu'après coup si on a gardé une copie)

print(f"Parcelles dans df_wide           : {len(df_wide):,}")
print(f"Parcelles avec >=1 date filtrée  : {len(parcelles_completes):,}")
print(f"Parcelles sans aucune date       : {len(parcelles_sans_date_valide):,}")
print(f"(dont connu S2.4, sans pixel 20m): 2 751 attendu)")

# --- Diagnostic : fiabilité des jour_max_ndvi extrêmes --------------------

candidats_suspects = df_ndvi_dates.loc[idx_max].merge(
    df_date_max[["jour_max_ndvi"]], left_on="id_parcel", right_index=True
)
suspects = candidats_suspects[candidats_suspects["jour_max_ndvi"] <= 15]

print(f"\nParcelles avec jour_max_ndvi <= 15 jours : {len(suspects):,}")
print(suspects[["mean", "std", "n_pixels"]].describe().to_string())

In [ ]:
# --- Pente NDVI mai → août (à partir des composites mensuels) -----------

JOURS_MAI_AOUT = (pd.Timestamp("2024-08-15") - pd.Timestamp("2024-05-15")).days

features_temporelles["pente_ndvi_mai_aout"] = (
    df_wide["NDVI_mean_2024-08"] - df_wide["NDVI_mean_2024-05"]
) / JOURS_MAI_AOUT

# --- Fusion dans df_wide (idempotent) ------------------------------------

NOUVELLES_FEATURES = ["amplitude_ndvi", "jour_max_ndvi", "pente_ndvi_mai_aout"]
df_wide = df_wide.drop(columns=NOUVELLES_FEATURES, errors="ignore")
df_wide = df_wide.join(features_temporelles[NOUVELLES_FEATURES])

print(f"df_wide : {df_wide.shape[1]} colonnes "
      f"({df_wide.shape[1] - 3 - 2} features S2 + 3 nouvelles + classe/split)")
print(f"NaN sur les nouvelles features : "
      f"{df_wide[NOUVELLES_FEATURES].isna().sum().to_dict()}")

In [ ]:
# --- Réentraînement rf_base sur le set augmenté --------------------------

feature_cols_aug = [c for c in df_wide.columns if c not in ("classe", "split")]

X_train_aug = df_wide.loc[train_mask, feature_cols_aug].values
X_test_aug = df_wide.loc[test_mask, feature_cols_aug].values

rf_aug = RandomForestClassifier(
    n_estimators=300, max_depth=30, min_samples_leaf=5,
    max_features="sqrt", class_weight="balanced",
    random_state=SEED, n_jobs=-1,
)
rf_aug.fit(X_train_aug, y_train)

y_pred_aug = rf_aug.predict(X_test_aug)

print(f"Accuracy train : {rf_aug.score(X_train_aug, y_train):.4f}")
print(f"Accuracy test  : {rf_aug.score(X_test_aug, y_test):.4f}")
print(f"\n{classification_report(y_test, y_pred_aug, digits=3)}")

# --- Comparaison ciblée : F1 'autres' et confusion avec 'prairie' -------

f1_autres_avant = 0.678  # run précédent (rf_base sans features temporelles)
report_aug = classification_report(y_test, y_pred_aug, digits=3, output_dict=True)
f1_autres_apres = report_aug["autres"]["f1-score"]

cm_aug = confusion_matrix(y_test, y_pred_aug, labels=sorted(np.unique(y_train)))
classes_sorted = sorted(np.unique(y_train))
i_autres, i_prairie = classes_sorted.index("autres"), classes_sorted.index("prairie")

print(f"\nF1 'autres'      : {f1_autres_avant:.3f} → {f1_autres_apres:.3f} "
      f"({f1_autres_apres - f1_autres_avant:+.3f})")
print(f"autres → prairie : {cm_aug[i_autres, i_prairie]} (référence : 578)")
print(f"prairie → autres : {cm_aug[i_prairie, i_autres]} (référence : 508)")

# --- Feature importances : les nouvelles features apparaissent-elles ? --

importances_aug = pd.Series(rf_aug.feature_importances_, index=feature_cols_aug)
rang_nouvelles = importances_aug.rank(ascending=False)[NOUVELLES_FEATURES]
print(f"\nRang des nouvelles features (sur {len(feature_cols_aug)}) :")
print(rang_nouvelles.astype(int).to_string())

### 4.4 — Sauvegarde des prédictions (préparation S5)

Persistance des prédictions du modèle final (`search.best_estimator_`)
dans une nouvelle table PostGIS, en prévision de l'API FastAPI (S5) qui
exposera par parcelle sa classe prédite et son score de confiance —
et, dans une itération ultérieure, son score de divergence (S4) et
ses métriques phénologiques (S4).

Le modèle est réappliqué à l'ensemble des parcelles (train + test),
pas seulement au jeu de test : l'objectif n'est plus l'évaluation
mais la couverture complète nécessaire à S4 et S5. La colonne `split`
est conservée pour distinguer une prédiction in-sample (train) d'une
prédiction out-of-sample (test).

In [ ]:
# --- Création de la table (idempotent) ----------------------------------

DDL_CLASSIFICATION = """
CREATE TABLE IF NOT EXISTS derived.parcelles_classification (
    id_parcel       TEXT PRIMARY KEY,
    classe_predite  TEXT NOT NULL,
    classe_declaree TEXT NOT NULL,
    proba_max       REAL NOT NULL,
    split           TEXT NOT NULL CHECK (split IN ('train', 'test')),
    model_version   TEXT NOT NULL,
    date_prediction TIMESTAMPTZ NOT NULL DEFAULT now()
);
"""
# ⚠ Vérifie que le type de id_parcel correspond à celui de
#   derived.rpg_parcelles_aoi.id_parcel (TEXT supposé ici).

conn = psycopg2.connect(**PG_PARAMS)
with conn.cursor() as cur:
    cur.execute(DDL_CLASSIFICATION)
conn.commit()
conn.close()

print("Table derived.parcelles_classification prête (créée si absente).")

In [ ]:
# --- Prédiction sur l'ensemble des parcelles (train + test) -------------

X_full = df_wide.loc[:, feature_cols].values
proba_full = rf.predict_proba(X_full)
y_pred_full = rf.classes_[np.argmax(proba_full, axis=1)]
proba_max_full = proba_full.max(axis=1)

MODEL_VERSION = "rf_tuned_" + pd.Timestamp.today().strftime("%Y%m%d")

df_predictions = pd.DataFrame({
    "id_parcel": df_wide.index,
    "classe_predite": y_pred_full,
    "classe_declaree": df_wide["classe"].to_numpy(),
    "proba_max": proba_max_full,
    "split": df_wide["split"].to_numpy(),
    "model_version": MODEL_VERSION,
})

print(f"Prédictions générées : {len(df_predictions):,} parcelles")
print(df_predictions["split"].value_counts().to_string())

In [ ]:
# --- Upsert dans PostGIS -------------------------------------------------

from psycopg2.extras import execute_values

records = list(
    df_predictions[
        ["id_parcel", "classe_predite", "classe_declaree", "proba_max", "split", "model_version"]
    ].itertuples(index=False, name=None)
)

sql_upsert = """
    INSERT INTO derived.parcelles_classification
        (id_parcel, classe_predite, classe_declaree, proba_max, split, model_version)
    VALUES %s
    ON CONFLICT (id_parcel) DO UPDATE SET
        classe_predite  = EXCLUDED.classe_predite,
        classe_declaree = EXCLUDED.classe_declaree,
        proba_max       = EXCLUDED.proba_max,
        split           = EXCLUDED.split,
        model_version   = EXCLUDED.model_version,
        date_prediction = now()
"""

conn = psycopg2.connect(**PG_PARAMS)
with conn.cursor() as cur:
    execute_values(cur, sql_upsert, records, page_size=5000)
conn.commit()
conn.close()

print(f"{len(records):,} prédictions upsertées dans derived.parcelles_classification")

In [ ]:
# --- Vérification ---------------------------------------------------------

conn = psycopg2.connect(**PG_PARAMS)
sql_check = """
    SELECT model_version, split, COUNT(*) AS n,
           ROUND(AVG(proba_max)::numeric, 3) AS proba_moy
    FROM derived.parcelles_classification
    GROUP BY model_version, split
    ORDER BY model_version, split
"""
df_check = pd.read_sql(sql_check, conn)
conn.close()

print(df_check.to_string(index=False))